# Проверки надёжности и воспроизводимости

**Часть 5.** Здесь мы проверяем выводы предыдущих частей на прочность. Четыре вопроса:

1. **Сегменты.** Опираются ли выводы о различиях эффекта на формальные тесты взаимодействия (а не только на отдельные тесты внутри сегментов), и выдерживают ли они поправки на множественные сравнения?
2. **Uplift-модели.** Не использует ли сравнение моделей и простого правила одни и те же данные для выбора и для оценки? Насколько устойчивы Qini и экономические выводы к случайному разбиению?
3. **Python и SQL.** Совпадают ли результаты двух реализаций?
4. **Числа в README.** Совпадают ли ключевые числа с пересчётом на чистом ядре?

Ноутбук самодостаточен: загружает данные и повторяет нужные функции предыдущих частей.

In [1]:
import glob, sys, platform, sqlite3, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import scipy, sklearn, statsmodels
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

# Версии окружения: фиксируем, на чём получены результаты
versions = {
    "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
    "scipy": scipy.__version__, "statsmodels": statsmodels.__version__, "scikit-learn": sklearn.__version__,
}
print(versions)

paths = glob.glob("/kaggle/input/**/*Hillstrom*.csv", recursive=True) \
     or glob.glob("/kaggle/input/**/*.csv", recursive=True) \
     or glob.glob("../archive/*.csv")
raw = pd.read_csv(paths[0])
print("Файл:", paths[0], raw.shape)

df = raw.copy()
df["newbie_label"] = df["newbie"].map({0: "existing", 1: "new"})
df["recency_bucket"] = pd.cut(df["recency"], [0, 3, 6, 9, 12], labels=["1-3", "4-6", "7-9", "10-12"]).astype(str)
df["buyer_type"] = np.select(
    [(df.mens == 1) & (df.womens == 0), (df.mens == 0) & (df.womens == 1), (df.mens == 1) & (df.womens == 1)],
    ["only_mens", "only_womens", "both"], default="neither")
CONTROL = "No E-Mail"
EMAILS = ["Mens E-Mail", "Womens E-Mail"]
DIMS = ["newbie_label", "channel", "zip_code", "history_segment", "recency_bucket", "buyer_type"]

{'python': '3.13.15', 'pandas': '2.3.3', 'numpy': '2.1.3', 'scipy': '1.16.3', 'statsmodels': '0.15.0', 'scikit-learn': '1.6.1'}
Файл: /kaggle/input/datasets/sumittarya/hillstrom-email-marketing/Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv (64000, 12)


## 1. Различается ли эффект между сегментами: формальные тесты

Раньше формальную проверку делали для визитов и конверсии. Здесь добавляем **траты** и показываем, как меняются выводы при разных способах поправки.

Для визитов и конверсии: логистическая регрессия «письмо × сегмент» и критерий отношения правдоподобия. Для трат (непрерывная, скошенная метрика): линейная модель с робастными ошибками HC3 и тест Вальда на все взаимодействия.

Поправка Бенджамини-Хохберга (FDR) применяется двумя способами:
- внутри каждой метрики (12 проверок: 2 письма × 6 признаков);
- сразу ко всем 36 проверкам (самый строгий вариант).

In [2]:
def interaction_p(metric, treat, dim):
    d = df[df.segment.isin([treat, CONTROL])].copy()
    d["treated"] = (d.segment == treat).astype(int)
    if metric in ("visit", "conversion"):
        m0 = smf.logit(f"{metric} ~ treated + C({dim})", d).fit(disp=0)
        m1 = smf.logit(f"{metric} ~ treated * C({dim})", d).fit(disp=0)
        return stats.chi2.sf(2 * (m1.llf - m0.llf), m1.df_model - m0.df_model)
    m1 = smf.ols(f"{metric} ~ treated * C({dim})", d).fit(cov_type="HC3")
    names = [n for n in m1.params.index if n.startswith("treated:")]
    R = np.zeros((len(names), len(m1.params)))
    for i, n in enumerate(names):
        R[i, list(m1.params.index).index(n)] = 1
    return float(m1.wald_test(R, scalar=True).pvalue)

rows = [(m, t, dim, interaction_p(m, t, dim)) for m in ["visit", "conversion", "spend"] for t in EMAILS for dim in DIMS]
inter = pd.DataFrame(rows, columns=["метрика", "письмо", "признак", "p-value"])
inter["q (внутри метрики)"] = inter.groupby("метрика")["p-value"].transform(lambda x: multipletests(x, method="fdr_bh")[1])
inter["q (все 36 проверок)"] = multipletests(inter["p-value"], method="fdr_bh")[1]
inter.sort_values("p-value").head(12).reset_index(drop=True)

,метрика,письмо,признак,p-value,q (внутри метрики),q (все 36 проверок)
0,visit,Womens E-Mail,buyer_type,0.0000,0.0000,0.0000
1,visit,Womens E-Mail,newbie_label,0.0001,0.0003,0.0010
2,visit,Mens E-Mail,newbie_label,0.0005,0.0019,0.0056
3,conversion,Womens E-Mail,newbie_label,0.0013,0.0152,0.0114
4,visit,Womens E-Mail,zip_code,0.0049,0.0148,0.0356
5,spend,Womens E-Mail,newbie_label,0.0177,0.2130,0.0985
6,visit,Womens E-Mail,recency_bucket,0.0191,0.0460,0.0985
7,conversion,Mens E-Mail,newbie_label,0.0268,0.1311,0.1107
8,visit,Mens E-Mail,zip_code,0.0277,0.0553,0.1107
9,conversion,Womens E-Mail,history_segment,0.0328,0.1311,0.1180


In [3]:
# Какие выводы о зависимости эффекта от сегмента выдерживают поправку
inter["значимо внутри метрики"] = inter["q (внутри метрики)"] < 0.05
inter["значимо среди всех 36"] = inter["q (все 36 проверок)"] < 0.05
inter[inter["значимо внутри метрики"] | inter["значимо среди всех 36"]].sort_values("p-value").reset_index(drop=True)

,метрика,письмо,признак,p-value,q (внутри метрики),q (все 36 проверок),значимо внутри метрики,значимо среди всех 36
0,visit,Womens E-Mail,buyer_type,0.0000,0.0000,0.0000,True,True
1,visit,Womens E-Mail,newbie_label,0.0001,0.0003,0.0010,True,True
2,visit,Mens E-Mail,newbie_label,0.0005,0.0019,0.0056,True,True
3,conversion,Womens E-Mail,newbie_label,0.0013,0.0152,0.0114,True,True
4,visit,Womens E-Mail,zip_code,0.0049,0.0148,0.0356,True,True
5,visit,Womens E-Mail,recency_bucket,0.0191,0.0460,0.0985,True,False


### Прямые сравнения эффектов

Фраза «в одном сегменте эффект значим, в другом нет» не доказывает, что эффекты различаются: разница между значимым и незначимым результатом сама может быть незначимой. Поэтому проверяем разницу эффектов напрямую, с доверительным интервалом (линейная модель с робастными ошибками). Это сравнения двух групп, выбранные после просмотра данных, поэтому p-value здесь без поправки.

In [4]:
def contrast(metric, flag_col, flag_name, data):
    d = data.copy()
    d["treated"] = (d.segment == "Womens E-Mail").astype(int)
    m = smf.ols(f"{metric} ~ treated * {flag_col}", d).fit(cov_type="HC3")
    key = f"treated:{flag_col}"
    lo, hi = m.conf_int().loc[key]
    scale = 100 if metric in ("visit", "conversion") else 1
    return {"сравнение": flag_name, "метрика": metric, "разница эффектов": m.params[key] * scale,
            "CI95 низ": lo * scale, "CI95 верх": hi * scale, "p-value": m.pvalues[key]}

wm = df[df.segment.isin(["Womens E-Mail", CONTROL])]
d_bt = wm[wm.buyer_type.isin(["only_womens", "only_mens"])].assign(ow=lambda x: (x.buyer_type == "only_womens").astype(int))
rows = []
for metric in ["visit", "conversion", "spend"]:
    rows.append(contrast(metric, "ow", "женское письмо: покупавшие женское минус покупавшие мужское", d_bt))
    rows.append(contrast(metric, "newbie", "женское письмо: новички минус существующие", wm))
pd.DataFrame(rows)

,сравнение,метрика,разница эффектов,CI95 низ,CI95 верх,p-value
0,женское письмо: покупавшие женское минус покуп...,visit,6.2907,5.0011,7.5803,0.0000
1,женское письмо: новички минус существующие,visit,1.0295,-0.2360,2.2950,0.1108
2,женское письмо: покупавшие женское минус покуп...,conversion,0.4482,0.1207,0.7757,0.0073
3,женское письмо: новички минус существующие,conversion,0.4340,0.1114,0.7567,0.0084
4,женское письмо: покупавшие женское минус покуп...,spend,0.3533,-0.1709,0.8775,0.1865
5,женское письмо: новички минус существующие,spend,0.6183,0.1072,1.1295,0.0177


Единицы: для визитов и конверсии разница в процентных пунктах, для трат в долларах на клиента.

## 2. Независимая оценка uplift-моделей

В части 3 лучшая модель выбиралась по Qini на тех же предсказаниях, на которых затем показывались децили и считалась экономика. Гиперпараметры при этом не подбирались, поэтому утечка небольшая, но формально оценка не независима. Здесь проверяем двумя способами:

- **Устойчивость к разбиению:** повторяем кросс-валидацию с пятью разными seed и смотрим разброс Qini.
- **Отложенная выборка:** 30% клиентов откладываем и не трогаем. Выбор модели и охват (долю клиентов, которым шлём письмо) делаем **только на остальных 70%**, затем один раз оцениваем на отложенных 30%. Повторяем для 10 случайных разбиений, чтобы увидеть разброс.

In [5]:
NUM = ["recency", "history", "mens", "womens", "newbie"]
CAT = ["zip_code", "channel"]
FEATURES = NUM + CAT
SEED = 42

def make_prep(scale):
    return ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
                              ("num", StandardScaler() if scale else "passthrough", NUM)])

def gb_clf():
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, min_samples_leaf=100, random_state=SEED)

def gb_reg():
    return HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05, max_iter=150, min_samples_leaf=100, random_state=SEED)

def lr_clf():
    return LogisticRegression(max_iter=2000, C=1.0)

def fit_predict(train, test, target, kind):
    if kind in ("t_lr", "t_gb", "t_gb_spend"):
        scale = kind == "t_lr"
        est = {"t_lr": lr_clf, "t_gb": gb_clf, "t_gb_spend": gb_reg}[kind]
        preds = {}
        for t in (0, 1):
            m = make_pipeline(make_prep(scale), est())
            sub = train[train["T"] == t]
            m.fit(sub[FEATURES], sub[target])
            preds[t] = m.predict(test[FEATURES]) if kind == "t_gb_spend" else m.predict_proba(test[FEATURES])[:, 1]
        return preds[1] - preds[0]
    if kind == "ct_lr":
        m = make_pipeline(make_prep(True), lr_clf())
        m.fit(train[FEATURES], (train[target] == train["T"]).astype(int))
        return 2 * m.predict_proba(test[FEATURES])[:, 1] - 1
    raise ValueError(kind)

def oof_scores(d, target, kind, seed, n_splits=5):
    skf = StratifiedKFold(n_splits, shuffle=True, random_state=seed)
    strat = d["T"].astype(str) + "_" + (d[target] > 0).astype(str)
    score = np.zeros(len(d))
    for tr, te in skf.split(d, strat):
        score[te] = fit_predict(d.iloc[tr], d.iloc[te], target, kind)
    return score

def qini_gap(y, t, score):
    order = np.argsort(-score, kind="stable")
    y, t = y[order], t[order]
    cum_t, cum_c = np.cumsum(t), np.cumsum(1 - t)
    q = np.cumsum(y * t) - np.cumsum(y * (1 - t)) * cum_t / np.maximum(cum_c, 1)
    frac = np.arange(1, len(y) + 1) / len(y)
    return np.mean(q - q[-1] * frac)

def profit_at(d, score, cost, grid):
    order = np.argsort(-score, kind="stable")
    x = d.iloc[order].reset_index(drop=True)
    out = []
    for k in grid:
        top = x.iloc[: int(round(k * len(x)))]
        up = top[top["T"] == 1]["spend"].mean() - top[top["T"] == 0]["spend"].mean()
        out.append(1000 * k * (up - cost))
    return np.array(out)

def rule_gain(d, cost):
    up_all = d[d["T"] == 1]["spend"].mean() - d[d["T"] == 0]["spend"].mean()
    m = d["womens"] == 1
    g = d[m]
    up = g[g["T"] == 1]["spend"].mean() - g[g["T"] == 0]["spend"].mean()
    return 1000 * m.mean() * (up - cost) - 1000 * (up_all - cost)

MODELS = {
    "T-learner, логрегрессия": ("t_lr", "visit"),
    "T-learner, бустинг": ("t_gb", "visit"),
    "Преобразование классов, логрегрессия": ("ct_lr", "visit"),
    "T-learner, бустинг по тратам": ("t_gb_spend", "spend"),
}
GRID = np.linspace(0.05, 1.0, 20)
COSTS = [0.05, 0.10, 0.20, 0.30, 0.40]

d_w = df[df.segment.isin(["Womens E-Mail", CONTROL])].reset_index(drop=True).copy()
d_w["T"] = (d_w.segment == "Womens E-Mail").astype(int)
len(d_w)

42693

### 2a. Устойчивость Qini к разбиению кросс-валидации
Qini выражен в числе дополнительных визитов (для трат в долларах). Чем меньше разброс между seed, тем надёжнее оценка.

In [6]:
rows = []
for seed in range(5):
    for name, (kind, target) in MODELS.items():
        s = oof_scores(d_w, target, kind, seed)
        rows.append({"seed": seed, "модель": name,
                     "Qini, визиты": qini_gap(d_w["visit"].to_numpy(float), d_w["T"].to_numpy(), s),
                     "Qini, траты": qini_gap(d_w["spend"].to_numpy(float), d_w["T"].to_numpy(), s)})
stab = pd.DataFrame(rows)
display(stab.groupby("модель")[["Qini, визиты", "Qini, траты"]].agg(["mean", "std", "min", "max"]))

visit_models = stab[stab["модель"] != "T-learner, бустинг по тратам"]
best_by_seed = visit_models.loc[visit_models.groupby("seed")["Qini, визиты"].idxmax(), ["seed", "модель"]]
best_by_seed

Qini, визиты                           Qini, траты                             
                                             mean     std      min      max        mean      std      min        max
модель                                                                                                              
T-learner, бустинг                       142.2165  4.2620 136.9823 148.8495    640.0380 427.8135 -27.8123 1,123.2244
T-learner, бустинг по тратам              13.8696 24.2165  -9.0347  55.1593    849.7339 342.1983 420.3924 1,241.0132
T-learner, логрегрессия                  153.9085  1.7985 150.7818 155.4154    647.5558 207.9673 348.3635   868.9113
Преобразование классов, логрегрессия     156.2787  4.9155 151.3722 163.5531    620.3822 214.8059 416.3992   895.2733

,seed,модель
2,0,"Преобразование классов, логрегрессия"
4,1,"T-learner, логрегрессия"
10,2,"Преобразование классов, логрегрессия"
14,3,"Преобразование классов, логрегрессия"
18,4,"Преобразование классов, логрегрессия"


### 2b. Отложенная выборка

In [7]:
res_q, res_gain = [], []
for rep in range(10):
    strat = d_w["T"].astype(str) + "_" + d_w["visit"].astype(str)
    dev, hold = train_test_split(d_w, test_size=0.3, stratify=strat, random_state=100 + rep)
    dev, hold = dev.reset_index(drop=True), hold.reset_index(drop=True)
    for name, (kind, target) in MODELS.items():
        s_dev = oof_scores(dev, target, kind, SEED)          # выбор охвата только по 70%
        s_hold = fit_predict(dev, hold, target, kind)        # модель обучена на 70%, оценка на 30%
        res_q.append({"разбиение": rep, "модель": name,
                      "Qini, визиты": qini_gap(hold["visit"].to_numpy(float), hold["T"].to_numpy(), s_hold),
                      "Qini, траты": qini_gap(hold["spend"].to_numpy(float), hold["T"].to_numpy(), s_hold)})
        for cost in COSTS:
            k_star = GRID[int(np.argmax(profit_at(dev, s_dev, cost, GRID)))]
            gain = profit_at(hold, s_hold, cost, [k_star])[0] - profit_at(hold, s_hold, cost, [1.0])[0]
            res_gain.append({"разбиение": rep, "модель": name, "стоимость письма": cost, "выигрыш": gain})
    for cost in COSTS:
        res_gain.append({"разбиение": rep, "модель": "Правило: слать, если покупал женское",
                         "стоимость письма": cost, "выигрыш": rule_gain(hold, cost)})
hold_q, hold_gain = pd.DataFrame(res_q), pd.DataFrame(res_gain)
print("Размер отложенной выборки:", len(hold))
display(hold_q.groupby("модель")[["Qini, визиты", "Qini, траты"]].agg(["mean", "std"]))

Размер отложенной выборки: 12808


Qini, визиты         Qini, траты         
                                             mean     std        mean      std
модель                                                                        
T-learner, бустинг                        38.8502  7.8135    455.5747 357.0051
T-learner, бустинг по тратам               9.6491 10.1039    299.6937 781.5052
T-learner, логрегрессия                   50.2203  9.7642    448.9596 371.6816
Преобразование классов, логрегрессия      48.9729  4.4869    445.9083 342.2421

In [8]:
print("Выигрыш относительно рассылки всем на отложенных 30%, $ на 1000 клиентов базы: среднее по 10 разбиениям")
display(hold_gain.pivot_table(index="модель", columns="стоимость письма", values="выигрыш", aggfunc="mean"))
print("Стандартное отклонение по разбиениям")
display(hold_gain.pivot_table(index="модель", columns="стоимость письма", values="выигрыш", aggfunc="std"))
print("Доля разбиений, где таргетинг выиграл у рассылки всем")
display(hold_gain.assign(выиграл=hold_gain["выигрыш"] > 0).pivot_table(index="модель", columns="стоимость письма", values="выиграл", aggfunc="mean"))

Выигрыш относительно рассылки всем на отложенных 30%, $ на 1000 клиентов базы: среднее по 10 разбиениям


стоимость письма,0.0500,0.1000,0.2000,0.3000,0.4000
модель,,,,,
"T-learner, бустинг",-14.6310,-7.5862,4.8139,53.6470,100.6585
"T-learner, бустинг по тратам",-27.7827,-30.6315,-44.4729,-30.1870,30.5156
"T-learner, логрегрессия",-12.4577,-3.1292,-8.9924,25.0415,74.2619
"Правило: слать, если покупал женское",-30.8786,-8.3310,36.7643,81.8595,126.9548
"Преобразование классов, логрегрессия",-33.1170,-30.3670,-29.7193,34.5538,49.5094


Стандартное отклонение по разбиениям


стоимость письма,0.0500,0.1000,0.2000,0.3000,0.4000
модель,,,,,
"T-learner, бустинг",69.2635,65.7789,73.2947,124.4708,127.2697
"T-learner, бустинг по тратам",82.4853,77.8367,144.1129,123.7725,146.2999
"T-learner, логрегрессия",75.5962,78.9887,92.3103,116.5975,119.6582
"Правило: слать, если покупал женское",164.0847,163.9960,163.8192,163.6431,163.4677
"Преобразование классов, логрегрессия",58.3097,54.2507,66.2048,94.1604,147.1034


Доля разбиений, где таргетинг выиграл у рассылки всем


стоимость письма,0.0500,0.1000,0.2000,0.3000,0.4000
модель,,,,,
"T-learner, бустинг",0.1000,0.3000,0.3000,0.5000,0.8000
"T-learner, бустинг по тратам",0.2000,0.2000,0.3000,0.4000,0.7000
"T-learner, логрегрессия",0.1000,0.2000,0.2000,0.4000,0.6000
"Правило: слать, если покупал женское",0.5000,0.5000,0.6000,0.6000,0.8000
"Преобразование классов, логрегрессия",0.0000,0.0000,0.1000,0.5000,0.7000


## 3. Совпадают ли Python и SQL
Загружаем данные в SQLite, выполняем запросы из папки `sql/` и сверяем результат с независимым расчётом в pandas.

In [9]:
SQL_SETUP = """
-- Таблица email_campaign загружается из CSV (Hillstrom Email Marketing).
-- Представление customers добавляет понятные сегменты.
DROP VIEW IF EXISTS customers;

CREATE VIEW customers AS
SELECT
    segment,
    recency,
    history,
    history_segment,
    zip_code,
    channel,
    newbie,
    mens,
    womens,
    CASE WHEN newbie = 1 THEN 'new' ELSE 'existing' END AS newbie_label,
    CASE
        WHEN recency <= 3 THEN '1-3'
        WHEN recency <= 6 THEN '4-6'
        WHEN recency <= 9 THEN '7-9'
        ELSE '10-12'
    END AS recency_bucket,
    CASE
        WHEN mens = 1 AND womens = 0 THEN 'only_mens'
        WHEN mens = 0 AND womens = 1 THEN 'only_womens'
        WHEN mens = 1 AND womens = 1 THEN 'both'
        ELSE 'neither'
    END AS buyer_type,
    visit,
    conversion,
    spend
FROM email_campaign;
"""

SQL_SUMMARY = """
-- Итоги теста по группам: размер, визиты, конверсия, траты
SELECT
    segment AS grp,
    COUNT(*) AS customers,
    ROUND(100.0 * AVG(visit), 2) AS visit_rate_pct,
    ROUND(100.0 * AVG(conversion), 3) AS conversion_rate_pct,
    ROUND(AVG(spend), 3) AS spend_per_customer,
    ROUND(SUM(spend) * 1.0 / NULLIF(SUM(conversion), 0), 2) AS avg_order_value
FROM customers
GROUP BY segment
ORDER BY CASE segment WHEN 'No E-Mail' THEN 0 WHEN 'Mens E-Mail' THEN 1 ELSE 2 END;
"""

SQL_FUNNEL = """
-- Воронка по группам: получили письмо -> зашли на сайт -> купили
SELECT
    segment AS grp,
    COUNT(*) AS sent,
    SUM(visit) AS visited,
    SUM(conversion) AS purchased,
    ROUND(100.0 * SUM(visit) / COUNT(*), 2) AS sent_to_visit_pct,
    ROUND(100.0 * SUM(conversion) / NULLIF(SUM(visit), 0), 2) AS visit_to_purchase_pct,
    ROUND(100.0 * SUM(conversion) / COUNT(*), 3) AS sent_to_purchase_pct
FROM customers
GROUP BY segment
ORDER BY CASE segment WHEN 'No E-Mail' THEN 0 WHEN 'Mens E-Mail' THEN 1 ELSE 2 END;
"""

SQL_SEGMENTS = """
-- Эффект каждого письма относительно контроля в каждом сегменте
WITH long AS (
    SELECT 'newbie_label' AS dimension, newbie_label AS value, segment, visit, conversion, spend FROM customers
    UNION ALL
    SELECT 'channel', channel, segment, visit, conversion, spend FROM customers
    UNION ALL
    SELECT 'zip_code', zip_code, segment, visit, conversion, spend FROM customers
    UNION ALL
    SELECT 'history_segment', history_segment, segment, visit, conversion, spend FROM customers
    UNION ALL
    SELECT 'recency_bucket', recency_bucket, segment, visit, conversion, spend FROM customers
    UNION ALL
    SELECT 'buyer_type', buyer_type, segment, visit, conversion, spend FROM customers
),
agg AS (
    SELECT
        dimension, value, segment,
        COUNT(*) AS n,
        AVG(visit) AS visit_rate,
        AVG(conversion) AS conv_rate,
        AVG(spend) AS spend_avg
    FROM long
    GROUP BY dimension, value, segment
)
SELECT
    t.dimension,
    t.value,
    t.segment AS email,
    t.n AS n_email,
    c.n AS n_control,
    ROUND(100.0 * c.visit_rate, 2) AS control_visit_pct,
    ROUND(100.0 * t.visit_rate, 2) AS email_visit_pct,
    ROUND(100.0 * (t.visit_rate - c.visit_rate), 2) AS visit_lift_pp,
    ROUND(100.0 * (t.conv_rate - c.conv_rate), 3) AS conversion_lift_pp,
    ROUND(t.spend_avg - c.spend_avg, 3) AS spend_lift
FROM agg t
JOIN agg c
    ON c.dimension = t.dimension AND c.value = t.value AND c.segment = 'No E-Mail'
WHERE t.segment <> 'No E-Mail'
ORDER BY t.dimension, t.value, t.segment;
"""

SQL_PROFIT = """
-- Вклад в прибыль на 1000 клиентов базы от женского письма
-- Параметры: {{ cost }} – стоимость одного письма, $; {{ margin }} – маржа с продаж (1 = 100% выручки)
-- spend в данных это выручка, поэтому вклад = margin * дополнительные траты - cost
-- Сумма строк = слать всем; строка bought_womens = 1 = правило "слать только покупавшим женское".
WITH base AS (
    SELECT
        womens AS bought_womens,
        COUNT(*) AS n,
        AVG(CASE WHEN segment = 'Womens E-Mail' THEN spend END) AS spend_email,
        AVG(CASE WHEN segment = 'No E-Mail' THEN spend END) AS spend_control
    FROM customers
    WHERE segment IN ('Womens E-Mail', 'No E-Mail')
    GROUP BY womens
)
SELECT
    bought_womens,
    n AS customers,
    ROUND(100.0 * n / (SELECT SUM(n) FROM base), 1) AS share_of_base_pct,
    ROUND(spend_email - spend_control, 3) AS spend_lift,
    ROUND(1000.0 * n / (SELECT SUM(n) FROM base)
          * ({{ margin }} * (spend_email - spend_control) - {{ cost }}), 1) AS profit_per_1000_if_sent
FROM base
ORDER BY bought_womens;
"""

In [10]:
con = sqlite3.connect(":memory:")
raw.to_sql("email_campaign", con, index=False)
con.executescript(SQL_SETUP)

def q(sql_text, **params):
    for k, v in params.items():
        sql_text = sql_text.replace("{{ " + k + " }}", str(v))
    return pd.read_sql(sql_text, con)

checks = []

# сводка по группам
s_sql = q(SQL_SUMMARY).set_index("grp")
s_pd = df.groupby("segment").agg(visit=("visit", "mean"), conv=("conversion", "mean"), spend=("spend", "mean"))
checks.append(("Сводка по группам", bool(
    np.allclose(s_sql["visit_rate_pct"], 100 * s_pd.loc[s_sql.index, "visit"], atol=0.006)
    and np.allclose(s_sql["conversion_rate_pct"], 100 * s_pd.loc[s_sql.index, "conv"], atol=0.0006)
    and np.allclose(s_sql["spend_per_customer"], s_pd.loc[s_sql.index, "spend"], atol=0.0006))))

# воронка
f_sql = q(SQL_FUNNEL).set_index("grp")
f_pd = df.groupby("segment").agg(sent=("visit", "size"), visited=("visit", "sum"), purchased=("conversion", "sum"))
checks.append(("Воронка (число клиентов на каждом шаге)", bool((f_sql[["sent", "visited", "purchased"]].values == f_pd.loc[f_sql.index].values).all())))

# эффекты по сегментам
seg_sql = q(SQL_SEGMENTS)
n_vals, n_bad = 0, 0
for _, r in seg_sql.iterrows():
    d = df[df[r["dimension"]].astype(str) == r["value"]]
    t, c = d[d.segment == r["email"]], d[d.segment == CONTROL]
    exp = {"visit_lift_pp": 100 * (t.visit.mean() - c.visit.mean()),
           "conversion_lift_pp": 100 * (t.conversion.mean() - c.conversion.mean()),
           "spend_lift": t.spend.mean() - c.spend.mean()}
    for k, v in exp.items():
        n_vals += 1
        n_bad += abs(r[k] - v) > (0.006 if k == "visit_lift_pp" else 0.0006)
checks.append((f"Эффекты по сегментам ({n_vals} значений)", n_bad == 0))

# вклад в прибыль
for margin in (1.0, 0.3):
    p = q(SQL_PROFIT, cost=0.10, margin=margin).set_index("bought_womens")["profit_per_1000_if_sent"]
    w = df[df.segment.isin(["Womens E-Mail", CONTROL])]
    ok = True
    for flag in (0, 1):
        g = w[w.womens == flag]
        up = g[g.segment == "Womens E-Mail"].spend.mean() - g[g.segment == CONTROL].spend.mean()
        ok &= abs(p.loc[flag] - 1000 * len(g) / len(w) * (margin * up - 0.10)) < 0.06
    checks.append((f"Вклад в прибыль, маржа {margin:.0%}", bool(ok)))

pd.DataFrame(checks, columns=["что сверяем", "Python == SQL"])

,что сверяем,Python == SQL
0,Сводка по группам,True
1,Воронка (число клиентов на каждом шаге),True
2,Эффекты по сегментам (132 значений),True
3,"Вклад в прибыль, маржа 100%",True
4,"Вклад в прибыль, маржа 30%",True


## 4. Совпадают ли ключевые числа README с пересчётом
Все значения ниже считаются заново из данных и сравниваются с числами, опубликованными в README и выводах.

In [11]:
conv = df.groupby("segment")["conversion"].mean() * 100
vis = df.groupby("segment")["visit"].mean() * 100
wm = df[df.segment.isin(["Womens E-Mail", CONTROL])]
def vis_lift(group):
    g = wm[wm.buyer_type == group]
    return 100 * (g[g.segment == "Womens E-Mail"].visit.mean() - g[g.segment == CONTROL].visit.mean())
g0 = wm[wm.womens == 0]
u_others = g0[g0.segment == "Womens E-Mail"].spend.mean() - g0[g0.segment == CONTROL].spend.mean()

claims = [
    ("Конверсия контроля, %", conv[CONTROL], 0.57, 0.005),
    ("Конверсия женского письма, %", conv["Womens E-Mail"], 0.88, 0.005),
    ("Конверсия мужского письма, %", conv["Mens E-Mail"], 1.25, 0.005),
    ("Рост конверсии женского письма, %", (conv["Womens E-Mail"] / conv[CONTROL] - 1) * 100, 54, 0.5),
    ("Мужское письмо: конверсия выше контроля в раз", conv["Mens E-Mail"] / conv[CONTROL], 2.2, 0.05),
    ("Визиты контроля, %", vis[CONTROL], 10.6, 0.05),
    ("Визиты мужского письма, %", vis["Mens E-Mail"], 18.3, 0.05),
    ("Визиты женского письма, %", vis["Womens E-Mail"], 15.1, 0.05),
    ("Женское письмо, прирост визитов у покупавших женское, п.п.", vis_lift("only_womens"), 7.4, 0.05),
    ("Женское письмо, прирост визитов у покупавших только мужское, п.п.", vis_lift("only_mens"), 1.1, 0.05),
    ("Допстраты группы 'покупали только мужское', $", u_others, 0.28, 0.005),
]
out = pd.DataFrame(claims, columns=["показатель", "пересчёт", "в README", "допуск"])
out["совпадает"] = (out["пересчёт"] - out["в README"]).abs() <= out["допуск"]
out

,показатель,пересчёт,в README,допуск,совпадает
0,"Конверсия контроля, %",0.5726,0.5700,0.0050,True
1,"Конверсия женского письма, %",0.8837,0.8800,0.0050,True
2,"Конверсия мужского письма, %",1.2531,1.2500,0.0050,True
3,"Рост конверсии женского письма, %",54.3313,54.0000,0.5000,True
4,Мужское письмо: конверсия выше контроля в раз,2.1884,2.2000,0.0500,True
5,"Визиты контроля, %",10.6167,10.6000,0.0500,True
6,"Визиты мужского письма, %",18.2757,18.3000,0.0500,True
7,"Визиты женского письма, %",15.1400,15.1000,0.0500,True
8,"Женское письмо, прирост визитов у покупавших ж...",7.3985,7.4000,0.0500,True
9,"Женское письмо, прирост визитов у покупавших т...",1.1078,1.1000,0.0500,True


## Выводы

**Окружение.** Результаты получены на Kaggle: Python 3.13.15, pandas 2.3.3, NumPy 2.1.3, SciPy 1.16.3, statsmodels 0.15.0, scikit-learn 1.6.1.

**1. Сегменты: что выдерживает поправку на множественные сравнения.**
- Подтверждено: зависимость эффекта женского письма на визиты от типа покупателя (q < 0,0001) и от статуса «новичок» (q = 0,0003); зависимость эффекта на конверсию от статуса «новичок» (q = 0,015 внутри метрики, 0,011 среди всех 36 проверок); зависимость эффекта мужского письма на визиты от статуса «новичок» (q = 0,0019, при почти одинаковых абсолютных эффектах).
- Зависимость женского письма на визиты от типа района значима внутри метрики (q = 0,015) и среди всех 36 проверок (q = 0,036); от давности значима только внутри метрики (q = 0,046, среди 36 уже нет).
- Не подтверждено: зависимость эффекта на конверсию от типа покупателя (q = 0,23) и любая зависимость эффекта на траты (лучший q = 0,21 внутри метрики, 0,10 среди всех 36).
- Прямые сравнения (без поправки): конверсия покупавших женское минус покупавших только мужское +0,45 п.п. [0,12; 0,78]; траты +$0,35 [−0,17; 0,88], то есть по тратам различие не доказано.

**2. Независимая оценка uplift-моделей.**
- Ранжирование по визитам подтверждается на отложенной выборке (30% клиентов, 10 разбиений): Qini 49–50 у логистических моделей и около 39 у бустинга.
- Лучшая модель по визитам нестабильна между запусками (преобразование классов в 4 из 5, T-learner с логрегрессией в 1 из 5), но логистические модели близки, а бустинг слабее.
- Qini по тратам очень шумный: стандартное отклонение по разбиениям 340–780 при среднем 300–460.
- При цене письма до $0,20 таргетинг не выигрывает у рассылки всем в большинстве разбиений (доля выигрышей 0–60%). При $0,30–0,40 средний выигрыш положителен у большинства стратегий, но стандартное отклонение (около 95–165 $) сопоставимо с самим выигрышем.
- Простое правило «покупал женское» не отличается от моделей по выигрышу в пределах разброса (при $0,30: +82 ± 164 против +54 ± 124 у бустинга).

**3. Python и SQL совпадают.** Сверены сводка по группам, воронка, 132 значения эффектов по сегментам и расчёт вклада в прибыль при марже 100% и 30%: расхождений нет.

**4. Числа в README совпадают с пересчётом.** Все 11 проверенных значений (конверсии, визиты, эффекты, допстраты) подтверждены.

**5. Воспроизводимость и среда.** Ноутбуки 1 и 2 при прогоне на разных компьютерах дали одинаковые числа. В ноутбуке 3 логистические модели воспроизводятся точно, а результаты бустинга зависят от среды (версии библиотек, потоки): на другом компьютере Qini по тратам для бустинга и экономические оценки отличались. Поэтому выводы по бустингу нужно читать как ориентир.

**Что это значит для рекомендации.** Устойчивы выводы по визитам и общему эффекту писем. Выводы о том, кому и при какой цене выгодно слать женское письмо, остаются предварительными: по покупкам и тратам данных недостаточно для надёжного таргетинга.